# RQ1 & RQ2 – Brain‑Tumor MRI Classification  
_Auto‑generated notebook – 2025-05-31 12:10 UTC_  

**Research questions**

1. **RQ1:** Which core preprocessing strategies (normalisation, resizing, histogram‑equalisation) yield the best baseline accuracy on heterogeneous MRI data?  
2. **RQ2:** How do data‑augmentation techniques (rotation, brightness/contrast, gamma, flip, crop, perspective warp, combined) plus different epoch budgets (5, 10, 15, 20) affect robustness?

All processing is done **in‑memory** (no cleaned images or CSVs written to disk).  


In [ ]:
# Environment & path setup
import os, sys, random, itertools, time
import numpy as np
import pandas as pd
import torch, torchvision
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import seaborn as sns

# project modules
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.append(ROOT)

from src.dataset import BrainTumorDataset
from src.experiment import Experiment
import torchvision.transforms as T

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

random.seed(0); torch.manual_seed(0)


In [ ]:
RAW_DATA_DIR = '../data/archive'  # Kaggle dataset location

def clahe_pil(img):
    """Apply CLAHE histogram equalisation to a PIL grayscale image."""
    arr = np.array(img, dtype=np.uint8)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    arr_eq = clahe.apply(arr)
    return Image.fromarray(arr_eq)

def build_preprocess_transform(image_size=160, zscore=True, clahe=False):
    """Return a torchvision transform implementing resize, optional CLAHE & z‑score."""
    steps = []
    if clahe:
        steps.append(T.Lambda(lambda im: clahe_pil(im)))
    steps.append(T.Resize((image_size, image_size)))
    steps.append(T.ToTensor())
    if zscore:
        steps.append(T.Lambda(lambda x: (x - x.mean()) / (x.std() + 1e-7)))
    return T.Compose(steps)

def attach_transform(exp, transform):
    # overwrite dataset transforms inside Experiment loaders
    exp.train_loader.dataset.transform = transform
    exp.val_loader.dataset.transform = transform

def train_for_epochs(exp, epochs):
    """Train experiment for *epochs* and return best validation accuracy & macro‑F1."""
    best_acc = 0.0
    for _ in range(epochs):
        exp._train_one_epoch()
        val_metrics = exp._evaluate()
        best_acc = max(best_acc, val_metrics['accuracy'])
    macro_f1 = exp._compute_class_metrics(val_metrics)['macro_f1']
    return best_acc, macro_f1


## RQ1 – Screening preprocessing factors

Grid:

| Factor | Values |
| ------ | ------- |
| Image size | 128, 160, 224 |
| Z‑score | off / on |
| CLAHE hist‑eq | off / on |

Each combo trains a 4‑class **SimpleCNN** for **5 epochs** (quick screen).


In [ ]:
IMG_SIZES = [128, 160, 224]
Z_OPTS    = [False, True]
HIST_OPTS = [False, True]

BATCH_SIZE = 16
LR = 1e-3
EPOCHS_RQ1 = 5

rq1_logs = []
for sz in IMG_SIZES:
    for zs in Z_OPTS:
        for he in HIST_OPTS:
            cfg = f'S{sz}_Z{int(zs)}_H{int(he)}'
            tf = build_preprocess_transform(sz, zscore=zs, clahe=he)

            exp = Experiment(
                data_root=RAW_DATA_DIR,
                model_variant='simple',
                input_size=sz,
                batch_size=BATCH_SIZE,
                lr=LR,
                epochs=EPOCHS_RQ1,
                disable_augment=True,
                device=DEVICE
            )
            exp.set_logger(lambda *_: None)
            exp.load_data(); exp._setup_optimizer_criterion()
            attach_transform(exp, tf)
            acc, m_f1 = train_for_epochs(exp, EPOCHS_RQ1)
            rq1_logs.append(dict(cfg=cfg, size=sz, zscore=zs, clahe=he,
                                 acc=acc, macro_f1=m_f1))
            print(cfg, '->', f'{acc:.2f}%')

rq1_df = pd.DataFrame(rq1_logs).sort_values('acc', ascending=False)
rq1_df


In [ ]:
pivot = rq1_df.pivot_table(values='acc', index='size', columns=['zscore','clahe'])
pivot.plot(kind='bar', figsize=(8,4))
plt.ylabel('Validation accuracy (%)')
plt.title('RQ1 – preprocessing grid (5 epochs)')
plt.show()

best_cfg = rq1_df.iloc[0]
BEST_PREPROC = dict(size=int(best_cfg['size']),
                    zscore=bool(best_cfg['zscore']),
                    clahe=bool(best_cfg['clahe']))
print('Best preprocessing combo:', BEST_PREPROC)


## RQ2 – Augmentation × Epoch budgets

Augmentation recipes:

| key | description |
|-----|-------------|
| none | no augmentation |
| rotate15 | ±15° rotation |
| bright_contrast | brightness/contrast jitter 0.8–1.2 |
| gamma | gamma jitter 0.8–1.2 |
| flip | horizontal flip |
| crop | random‑resized crop |
| perspective | random perspective warp |
| combo | all of the above |

Each recipe is trained with **5, 10, 15, 20** epochs.


In [ ]:
def build_aug_transform(base_size, pre_tf, mode):
    """Merge augmentation *mode* with the fixed preprocessing *pre_tf*."""
    aug_list = []
    if mode == 'rotate15':
        aug_list.append(T.RandomRotation(degrees=15))
    elif mode == 'bright_contrast':
        aug_list.append(T.ColorJitter(brightness=(0.8,1.2), contrast=(0.8,1.2)))
    elif mode == 'gamma':
        aug_list.append(T.RandomGamma((0.8,1.2)))
    elif mode == 'flip':
        aug_list.append(T.RandomHorizontalFlip(p=0.5))
    elif mode == 'crop':
        aug_list.append(T.RandomResizedCrop(size=base_size, scale=(0.8,1.0)))
    elif mode == 'perspective':
        aug_list.append(T.RandomPerspective(distortion_scale=0.2, p=0.5))
    elif mode == 'combo':
        aug_list.extend([
            T.RandomRotation(15),
            T.ColorJitter(brightness=(0.8,1.2), contrast=(0.8,1.2)),
            T.RandomGamma((0.8,1.2)),
            T.RandomHorizontalFlip(0.5),
            T.RandomResizedCrop(size=base_size, scale=(0.8,1.0)),
            T.RandomPerspective(0.2, p=0.5)
        ])
    # combine with preprocessing (ToTensor, z‑score etc.)
    return T.Compose(aug_list + pre_tf.transforms)


In [ ]:
AUG_RECIPES = ['none','rotate15','bright_contrast','gamma','flip','crop','perspective','combo']
EPOCHS_LIST = [5, 10, 15, 20]

base_sz = BEST_PREPROC['size']
prep_tf = build_preprocess_transform(base_sz,
                                     zscore=BEST_PREPROC['zscore'],
                                     clahe=BEST_PREPROC['clahe'])

rows = []
for aug in AUG_RECIPES:
    for epochs in EPOCHS_LIST:
        tf = prep_tf if aug=='none' else build_aug_transform(base_sz, prep_tf, aug)

        exp = Experiment(
            data_root=RAW_DATA_DIR,
            model_variant='simple',
            input_size=base_sz,
            batch_size=16,
            lr=1e-3,
            epochs=epochs,
            disable_augment=True,
            device=DEVICE
        )
        exp.set_logger(lambda *_: None)
        exp.load_data(); exp._setup_optimizer_criterion()
        attach_transform(exp, tf)

        acc, m_f1 = train_for_epochs(exp, epochs)
        rows.append(dict(aug=aug, epochs=epochs, acc=acc, macro_f1=m_f1))
        print(f'{aug:12s} | {epochs:2d} ep -> {acc:5.2f}%')

rq2_df = pd.DataFrame(rows)
rq2_df_pivot = rq2_df.pivot_table(values='acc', index='epochs', columns='aug')
rq2_df


In [ ]:
plt.figure(figsize=(10,4))
sns.heatmap(rq2_df_pivot, annot=True, fmt='.1f', cmap='viridis')
plt.title('Validation accuracy – augmentation vs epochs')
plt.show()

from scipy.stats import f_oneway
for ep in EPOCHS_LIST:
    vals = [rq2_df[(rq2_df['aug']==k)&(rq2_df['epochs']==ep)]['acc'].values
            for k in AUG_RECIPES]
    if any(len(v)==0 for v in vals):  # safety
        continue
    F,p = f_oneway(*vals)
    print(f'Epoch {ep}: ANOVA F={F:.2f}, p={p:.4g}')


### Discussion

* **RQ1:** The best preliminary preprocessing combination is printed above (`BEST_PREPROC`).  
* **RQ2:** Heat‑map + ANOVA indicate which augmentation / epoch combos significantly improve performance.  
Longer budgets allow stronger augmentations (crop/perspective/combo) to overtake light ones.

> Extend with cross‑validation, deeper models, or external datasets for tighter confidence.
